# **Visualizing Life Expectancy & Fertility (1964–2013)**

**Main Research Question:**
How did the relationship between fertility and life expectancy evolve across countries and continents between 1964 and 2013, and to what extent did countries converge toward lower fertility and higher life expectancy?

This question focuses on both temporal change and structural change. It examines not only how fertility and life expectancy changed independently, but also how the relationship between the two evolved over time.


In [32]:
import pandas as pd
import numpy as np
import plotly.express as px

In [22]:
df = pd.read_csv("gapminder.csv")
df.head()

# inspect data
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

# delete missing values
df = df.dropna(subset = ["lifeExp", "Fertility", "pop", "ID"])

(12200, 7)
Index(['Country', 'Year', 'lifeExp', 'pop', 'Fertility', 'Region', 'ID'], dtype='str')
Country          str
Year           int64
lifeExp      float64
pop          float64
Fertility    float64
Region           str
ID               str
dtype: object
Country         0
Year            0
lifeExp      2089
pop           394
Fertility    2100
Region          0
ID             50
dtype: int64
               Year       lifeExp           pop     Fertility
count  12200.000000  10111.000000  1.180600e+04  10100.000000
mean    1988.500000     64.078600  2.196105e+07      4.028719
std       14.431461     11.122779  9.593930e+07      2.013968
min     1964.000000      6.000000  5.000000e+01      0.836000
25%     1976.000000     56.282500  2.516718e+05      2.175750
50%     1988.500000     67.157000  3.497884e+06      3.632500
75%     2001.000000     72.484000  1.139999e+07      5.905250
max     2013.000000     83.580000  1.359368e+09      9.223000


## Analyze the Time Trend of Fertility and Life Expectancy

Calculate the average life expectancy and fertility of each year to see the change.

In [16]:
# analyze by year
global_trend = (
    df.groupby("Year").agg(
        ave_lifeExp = ("lifeExp", "mean"),
        ave_Fertility = ("Fertility", "mean")
    )
    .reset_index()
)
global_trend.head()

# calculate the overall change of life expectancy and fertility from 1964 to 2013
start = global_trend.iloc[0]
end = global_trend.iloc[-1]

print("Life expectancy change by:",
      round(end["ave_lifeExp"] - start["ave_lifeExp"],2))
print("Fertility change by:",
      round(end["ave_Fertility"] - start["ave_Fertility"],2))

Life expectancy change by: 14.72
Fertility change by: -2.65


Compare the trend of different continents of each year

In [19]:
# analyze by region and year
region_trend = (
    df.groupby(["Year","Region"]).agg(
        ave_lifeExp = ("lifeExp", "mean"),
        ave_Fertility = ("Fertility", "mean")
    )
    .reset_index()
)
region_trend.head()

,Year,Region,ave_lifeExp,ave_Fertility
0,1964,America,60.462775,5.574650
1,1964,East Asia & Pacific,56.798429,5.708032
2,1964,Europe & Central Asia,67.840110,3.270488
3,1964,Middle East & North Africa,52.119810,6.965571
4,1964,South Asia,43.877125,6.480500


Analyze the correlation coefficient of the two variables

In [21]:
# Analyze the relationship between the two variables using the whole sample
overall_correlation = df["Fertility"].corr(df["lifeExp"])
print("Overall correlation:",round(overall_correlation,2))

# the correlation coefficient is -0.83, really strong negative relation

Overall correlation: -0.83


In [43]:
# calculate the correlation coefficient of each year, regardless of continents and countries
corr_by_year = (
    df.groupby("Year")
    .apply(lambda x: x["Fertility"].corr(x["lifeExp"]))
    .reset_index(name = "Correlation")
)

print(corr_by_year.head(50))

fig = px.line(
    corr_by_year,
    x = "Year",
    y = "Correlation",
    title = "Correlation Between Fertility and Life Expectancy Over Time",
    markers = True
)

fig.add_hline(
    y = -0.70,
    line_dash = "dash",
    line_color = "black"
)

fig.show()

# we can see an obvious decline from 1964 to 1993 and a subtle increase from 1995 to 2013, but in this period the correlation is still strongly negative. We can also observe some outliers, 1975-1979 and 1994

    Year  Correlation
0   1964    -0.733581
1   1965    -0.737633
2   1966    -0.744933
3   1967    -0.754333
4   1968    -0.758072
5   1969    -0.763497
6   1970    -0.775376
7   1971    -0.781954
8   1972    -0.787504
9   1973    -0.789831
10  1974    -0.789870
11  1975    -0.777397
12  1976    -0.774055
13  1977    -0.775092
14  1978    -0.778061
15  1979    -0.784886
16  1980    -0.796251
17  1981    -0.811528
18  1982    -0.821979
19  1983    -0.827981
20  1984    -0.833540
21  1985    -0.837404
22  1986    -0.845601
23  1987    -0.850139
24  1988    -0.853624
25  1989    -0.854909
26  1990    -0.854884
27  1991    -0.854609
28  1992    -0.854107
29  1993    -0.849742
30  1994    -0.813271
31  1995    -0.844186
32  1996    -0.845832
33  1997    -0.843573
34  1998    -0.839094
35  1999    -0.833541
36  2000    -0.829154
37  2001    -0.825393
38  2002    -0.822217
39  2003    -0.819511
40  2004    -0.817615
41  2005    -0.814286
42  2006    -0.813493
43  2007    -0.813311
44  2008  

Calculate country-wide differences (convergence)

In [51]:
dispersion = (
    df.groupby("Year")
    .agg(
        lifeExp_std = ("lifeExp", "std"),
        Fertility_std = ("Fertility", "std")
    )
    .reset_index()
)

dispersion.head(50)

fig1 = px.line(
    dispersion,
    x = "Year",
    y = "lifeExp_std",
    title = "Change of std of life expectancy",
    markers = True
)
fig1.show()

fig2 = px.line(
    dispersion,
    x = "Year",
    y = "Fertility_std",
    title = "Change of std of Fertility",
    markers = True
)
fig2.show()

Recognize the countries with the largest change

In [55]:
country_change = (
    df[df["Year"].isin([1964,2013])]
    .pivot(
        index = "Country",
        columns = "Year",
        values = ["lifeExp","Fertility"])
)

country_change["lifeExp_change"] = (
    country_change[("lifeExp", 2013)] -
    country_change[("lifeExp", 1964)]
)

country_change["fertility_change"] = (
    country_change[("Fertility", 2013)] -
    country_change[("Fertility", 1964)]
)

country_change.sort_values(
    "lifeExp_change",
    ascending=False
).head(10)
#
# country_change.sort_values(
#     "fertility_change",
#     ascending=False
# ).head(10)

lifeExp         Fertility        lifeExp_change  \
Year              1964    2013      1964   2013                  
Country                                                          
Maldives        38.911  77.919     7.179  2.256         39.008   
Bhutan          33.827  68.294     6.670  2.232         34.467   
Timor-Leste     35.724  67.538     6.347  5.855         31.814   
Tunisia         44.903  75.873     7.107  2.008         30.970   
Oman            45.895  76.552     7.263  2.853         30.657   
Cambodia        41.900  71.916     6.909  2.861         30.016   
Nepal           40.071  68.410     5.997  2.300         28.339   
Western Sahara  39.880  67.764     6.562  2.363         27.884   
Saudi Arabia    47.781  75.479     7.257  2.644         27.698   
Afghanistan     33.639  60.947     7.671  4.900         27.308   

               fertility_change  
Year                             
Country                          
Maldives                 -4.923  
Bhutan                   -4.438  
Timor-Leste              -0.492  
Tunisia                  -5.099  
Oman                     -4.410  
Cambodia                 -4.048  
Nepal                    -3.697  
Western Sahara           -4.199  
Saudi Arabia             -4.613  
Afghanistan              -2.771